# Chain-KdV Visualization

Runs a single chain simulation and compares it with the KdV spectral solution.

- **Part A** – Chain (FPU lattice) integrated with Störmer-Verlet, reconstructed in co-moving frame  
- **Part B** – KdV solved with ETDRK4 Fourier spectral method  

See `CHAIN_KDV_MAPPING.md` for the full derivation of the correspondence.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rootutils
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

rootutils.setup_root(".", indicator=".project-root", pythonpath=True)
from data_generation.fput.generate import (  # noqa: E402
    ChainKdVIC,
    simulate_chain_kdv,
)

In [ ]:
# ── Simulation parameters ──────────────────────────────────────────────────────
A_KDV = 1.0  # KdV nonlinear coefficient
B_KDV = 1.0 / 24.0  # KdV dispersion coefficient (c = 24*b = 1)
LX = 2 * np.pi  # periodic domain length
N_CHAIN = 64  # FPU chain sites
N_KDV = 256  # output grid resolution
DT_KDV = 1e-3  # slow-time step
T = 1.0  # total slow time
CHAIN_SUBSTEPS = 50  # lab-time substeps per slow step
SAVE_EVERY = 10  # save frequency
NK = 3  # Fourier modes in IC
MAX_K = 6  # max wavenumber in IC

EPS = LX / N_CHAIN  # small parameter

# ── Build a random IC ─────────────────────────────────────────────────────────
rng_ic = np.random.default_rng(42)
u_ic = ChainKdVIC(LX, Nk=NK, max_k=MAX_K)
u_ic.reset(rng_ic)
print("IC:", u_ic)
print(f"eps = {EPS:.5f}  c = {24.0 * B_KDV:.4f}  alpha = {A_KDV * 24.0 * B_KDV:.4f}")

In [ ]:
# ── Run simulation ─────────────────────────────────────────────────────────────
common_kwargs = dict(
    a_kdv=A_KDV,
    b_kdv=B_KDV,
    eps=EPS,
    Lx=LX,
    N_chain=N_CHAIN,
    N_kdv=N_KDV,
    dt_kdv=DT_KDV,
    T=T,
    chain_substeps=CHAIN_SUBSTEPS,
    save_every=SAVE_EVERY,
)

t_slow, xi_grid, u_series = simulate_chain_kdv(u_ic, **common_kwargs)

print(f"u_series shape : {u_series.shape}  (frames, N_kdv)")
print(f"t_slow range   : [{t_slow[0]:.4f}, {t_slow[-1]:.4f}]")
print(f"u range        : [{u_series.min():.3f}, {u_series.max():.3f}]")

In [ ]:
# ── Static snapshots: initial, middle, final ──────────────────────────────────
n_frames = u_series.shape[0]
snap_idx = [0, n_frames // 2, n_frames - 1]

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, idx in zip(axes, snap_idx):
    ax.plot(xi_grid, u_series[idx], lw=1.5)
    ax.set_title(rf"$\tau$ = {t_slow[idx]:.4f}")
    ax.set_xlabel(r"$\xi$")
    ax.set_xlim(xi_grid[0], xi_grid[-1])
axes[0].set_ylabel(r"$u(\xi,\tau)$")
fig.suptitle("Chain-KdV: snapshots in co-moving frame", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── Animation ──────────────────────────────────────────────────────────────────
def animate_kdv(u_series, xi_grid, t_slow, *, interval=60):
    """HTML animation of u(xi, tau) evolving in slow time."""
    fig, ax = plt.subplots(figsize=(8, 4))
    (line,) = ax.plot(xi_grid, u_series[0], lw=1.5, color="C0")
    title = ax.set_title(rf"$\tau$ = {t_slow[0]:.4f}")
    ax.set_xlabel(r"$\xi$")
    ax.set_ylabel(r"$u(\xi,\tau)$")
    ax.set_xlim(xi_grid[0], xi_grid[-1])
    ypad = 0.05 * (u_series.max() - u_series.min())
    ax.set_ylim(u_series.min() - ypad, u_series.max() + ypad)
    fig.tight_layout()

    def update(frame):
        line.set_ydata(u_series[frame])
        title.set_text(rf"$\tau$ = {t_slow[frame]:.4f}")
        return line, title

    ani = FuncAnimation(fig, update, frames=u_series.shape[0], interval=interval, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())


animate_kdv(u_series, xi_grid, t_slow)

In [ ]:
# ── Space-time heatmap ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(
    u_series,
    aspect="auto",
    origin="lower",
    extent=[xi_grid[0], xi_grid[-1], t_slow[0], t_slow[-1]],
    cmap="RdBu_r",
)
ax.set_xlabel(r"$\xi$")
ax.set_ylabel(r"$\tau$ (slow time)")
ax.set_title(r"Space-time plot of $u(\xi,\tau)$")
fig.colorbar(im, ax=ax, label=r"$u$")
fig.tight_layout()
plt.show()